In [3]:
#!git clone https://github.com/neurovlm/neurovlm.git
#!cd neurovlm && pip install -e .
#!git clone https://github.com/neurovlm/neurovlm_data.git
from neurovlm.data import data_dir
print(data_dir)

import neurovlm
from neurovlm import data
from neurovlm.data import load_dataset
from neurovlm.models import load_model

import torch
import nibabel as nib
from nilearn import maskers
import numpy as np
from neurovlm.data import data_dir, load_masker, load_latent
from neurovlm.train import which_device
from nilearn.plotting import view_img
from nilearn.image import resample_img
import pandas as pd 
import pickle, gzip, numpy as np, sys

/Users/aaryapatel/.cache/neurovlm


In [16]:
# VISUAL NETWORK
device = which_device()
device = "cpu"
networks = load_dataset("networks")

# load first image
img = nib.Nifti1Image(networks["Du"]["VIS-P"]["array"], affine=networks["Du"]["VIS-P"]["affine"])
#img = nib.Nifti1Image((atlas_arr == 1).astype(np.float64), affine=atlas_img.affine)

def network():
    autoencoder = load_model("autoencoder").to(device)
    masker = load_masker()
    ###
    img_resampled = resample_img(img, masker.mask_img.affine)

    img_tensor = torch.from_numpy(
        masker.transform(img_resampled)
    )

    img_latent = autoencoder.encoder(img_tensor).cpu()

    latent_text, pmids_text = load_latent("publications")
    df = load_dataset("publications")

    inds = pd.Series(pmids_text).isin(df["pmid"])
    latent_text = latent_text[inds]
    latent_text = latent_text / latent_text.norm(dim=1)[:, None]
    pmids_text = pmids_text[inds]

    df = df.sort_values(by="pmid")

    ###
    proj_head = load_model("proj_head_text_mse").to(device)
    proj_head1= load_model("proj_head_text_infonce").to(device)
    proj_head2= load_model("proj_head_image_infonce").to(device)

    latent_text_aligned = proj_head1(latent_text)
    img_latent = proj_head2(img_latent)
    img_latent = img_latent / img_latent.norm()
    latent_text_aligned = latent_text_aligned / latent_text_aligned.norm(dim=1)[:, None]
    cos_sim = (img_latent @ latent_text_aligned.T)
    inds = torch.argsort(cos_sim, descending=True)


    #df = df[df["pmid"].isin(pmids_text)]
    ##
    print("\nTop 5 pubmed papers:")
    top_papers=list(df.iloc[inds].iloc[:5]["name"])
    i = 1
    for paper in top_papers:
        print(i, paper)
        i += 1
    
network()


networks_unpacked = []
for k, v in networks.items():
    for atlas_img_name, arr in v.items():
        networks_unpacked.append((k, atlas_img_name, nib.Nifti1Image(arr["array"], arr["affine"])))



/var/folders/7w/n8_hbb5j3dg0gtc5cwfcdqf80000gn/T/ipykernel_1008/2753255686.py:14: UserWarning: Resampling binary images with continuous or linear interpolation. This might lead to unexpected results. You might consider using nearest interpolation instead.
  img_resampled = resample_img(img, masker.mask_img.affine)



Top 5 pubmed papers:
1 Reorganization of early visual cortex functional connectivity following selective peripheral and central visual loss
2 Intrinsic Functional Connectivity Alterations of the Primary Visual Cortex in Primary Angle-Closure Glaucoma Patients before and after Surgery: A Resting-State fMRI Study.
3 Exploration of static functional connectivity and dynamic functional connectivity alterations in the primary visual cortex among patients with high myopia via seed-based functional connectivity analysis
4 Effects of primary angle-closure glaucoma on interhemispheric functional connectivity
5 Typical resting-state activity of the brain requires visual input during an early sensitive period


In [5]:
from neurovlm.data import data_dir

In [ ]:
# DEFAULT MODE NETWORK 

device = which_device()
device = "cpu"

networks = load_dataset("networks")

# Default Mode sub-networks
dmn_keys = ["DN-A", "DN-B"]
combined_dmn_data = None

for key in dmn_keys:
    arr = networks["Du"][key]["array"] > 0
    if combined_dmn_data is None:
        combined_dmn_data = arr
    else:
        combined_dmn_data |= arr 

img = nib.Nifti1Image(
    combined_dmn.astype(float),
    affine=networks["Du"][dmn_keys[0]]["affine"]
)

network()


/var/folders/7w/n8_hbb5j3dg0gtc5cwfcdqf80000gn/T/ipykernel_1008/2753255686.py:14: UserWarning: Resampling binary images with continuous or linear interpolation. This might lead to unexpected results. You might consider using nearest interpolation instead.
  img_resampled = resample_img(img, masker.mask_img.affine)



Top 5 pubmed papers:
1 Intrinsic Connectivity Networks in the Self- and Other-Referential Processing
2 Patterns of brain activity supporting autobiographical memory, prospection, and theory of mind, and their relationship to the default mode network.
3 The Functional Convergence and Heterogeneity of Social, Episodic, and Self-Referential Thought in the Default Mode Network
4 The development of the intrinsic functional connectivity of default network subsystems from age 3 to 5.
5 Default network connectivity in medial temporal lobe amnesia.


In [8]:
# Loop for all networks, but the user has to be very specific with the inputs:

import torch
import nibabel as nib
import numpy as np
from nilearn.image import resample_img

available_networks = [
    "VIS-P", "CG-OP", "DN-B", "SMOT-B", "AUD", "PM-PPr", "dATN-B", "SMOT-A", "LANG", "FPN-B", "FPN-A", "dATN-A",
    "VIS-C", "SAL/PMN", "DN-A", "NONE"
    ]

print("Available Networks in Du Atlas:")
i = 1
for net in available_networks:
    print(f"{i}. {net}")
    i += 1
choice = input("\nEnter the name of the network: ").strip()

if choice not in available_networks:
    print("Network name not found.")
else:
    print(f"\nProcessing your choice!")

    img = nib.Nifti1Image(
        networks["Du"][choice]["array"],
        affine=networks["Du"][choice]["affine"]
    )

    network()

Available Networks in Du Atlas:
1. VIS-P
2. CG-OP
3. DN-B
4. SMOT-B
5. AUD
6. PM-PPr
7. dATN-B
8. SMOT-A
9. LANG
10. FPN-B
11. FPN-A
12. dATN-A
13. VIS-C
14. SAL/PMN
15. DN-A
16. NONE

Processing your choice!


/var/folders/7w/n8_hbb5j3dg0gtc5cwfcdqf80000gn/T/ipykernel_1008/2753255686.py:14: UserWarning: Resampling binary images with continuous or linear interpolation. This might lead to unexpected results. You might consider using nearest interpolation instead.
  img_resampled = resample_img(img, masker.mask_img.affine)



Top 5 pubmed papers:
1 The processing of temporal pitch and melody information in auditory cortex.
2 Heschl's gyrus, posterior superior temporal gyrus, and mid-ventrolateral prefrontal cortex have different roles in the detection of acoustic changes.
3 Hierarchical processing of sound location and motion in the human brainstem and planum temporale.
4 Brain bases for auditory stimulus-driven figure-ground segregation.
5 Dichotic pitch activates pitch processing centre in Heschl's gyrus.


In [15]:
#LOOP CODE FOR ALL NETWORKS BUT WITH T=COMMON NETWORK NAME ALSO AS AN OPTION

import torch
import nibabel as nib
import numpy as np
from nilearn.image import resample_img

# Atlas keys

available_networks = [
    "VIS-P", "CG-OP", "DN-B", "SMOT-B", "AUD", "PM-PPr", "dATN-B", 
    "SMOT-A", "LANG", "FPN-B", "FPN-A", "dATN-A",
    "VIS-C", "SAL/PMN", "DN-A", "NONE"
]

# Common atlas keys

def get_prefix(name):
    if "-" in name:
        return name.split("-")[0]
    if "/" in name:
        return name.split("/")[0]
    return name 

groups = {}
for net in available_networks:
    prefix = get_prefix(net)
    groups.setdefault(prefix, []).append(net)

# Displaying the menu

print("Available Networks in Du Atlas:\n")
group_number = 1

for prefix, nets in groups.items():
    print(f"{group_number}. {prefix}:")
    for n in nets:
        print(f"   - {n}")
    print()
    group_number += 1


choice = input("\nEnter specific network name or the general name (e.g., VIS or VIS-P): ").strip()

selected_networks = []

if choice in available_networks:
    selected_networks = [choice]

elif choice in groups:
    selected_networks = groups[choice]

else:
    lowered = choice.lower()
    for prefix in groups:
        if lowered == prefix.lower():
            selected_networks = groups[prefix]
            break

if not selected_networks:
    print("Network not found.")
else:
    for s in selected_networks:
        print("  -", s)

    combined = None

    for s in selected_networks:
        arr = networks["Du"][s]["array"] > 0
        if combined is None:
            combined = arr.copy()
        else:
            combined |= arr  

    img = nib.Nifti1Image(
        combined.astype(float),
        affine=networks["Du"][selected_networks[0]]["affine"]
    )

    network()


Available Networks in Du Atlas:

1. VIS:
   - VIS-P
   - VIS-C

2. CG:
   - CG-OP

3. DN:
   - DN-B
   - DN-A

4. SMOT:
   - SMOT-B
   - SMOT-A

5. AUD:
   - AUD

6. PM:
   - PM-PPr

7. dATN:
   - dATN-B
   - dATN-A

8. LANG:
   - LANG

9. FPN:
   - FPN-B
   - FPN-A

10. SAL:
   - SAL/PMN

11. NONE:
   - NONE

  - AUD


/var/folders/7w/n8_hbb5j3dg0gtc5cwfcdqf80000gn/T/ipykernel_1008/2753255686.py:14: UserWarning: Resampling binary images with continuous or linear interpolation. This might lead to unexpected results. You might consider using nearest interpolation instead.
  img_resampled = resample_img(img, masker.mask_img.affine)



Top 5 pubmed papers:
1 The processing of temporal pitch and melody information in auditory cortex.
2 Heschl's gyrus, posterior superior temporal gyrus, and mid-ventrolateral prefrontal cortex have different roles in the detection of acoustic changes.
3 Hierarchical processing of sound location and motion in the human brainstem and planum temporale.
4 Brain bases for auditory stimulus-driven figure-ground segregation.
5 Dichotic pitch activates pitch processing centre in Heschl's gyrus.
